# Model checks

Supporting analyses for the README. *Saved outputs are from October 2026, before starting QBs
were taken from play-by-play; rerun to refresh. The conclusions didn't change (the Hybrid
backtest is 10.02 either way).*

In [ ]:
%load_ext autoreload
%autoreload 2

## How much does the gap to Vegas vary week to week?

Used to judge whether a bad live week is unusual: the weekly gap has a standard deviation of
about 0.9 points, so single weeks are mostly noise.

In [2]:
import pandas as pd
from nflmodel.config import BACKTEST_SEASONS
from nflmodel.features.build import build_features
from nflmodel.models.hybrid import HybridModel

games = build_features()
games = games[games["result"].notna() & games["spread_line"].notna()]

parts = []
for season in BACKTEST_SEASONS:
    train = games[games["season"] < season]
    test = games[games["season"] == season].copy()
    model = HybridModel()
    model.fit(train)
    test["pred"] = model.predict(test)
    parts.append(test)

bt = pd.concat(parts)

bt["gap"] = (bt["pred"] - bt["result"]).abs() - (bt["spread_line"] - bt["result"]).abs()
print(f"Overall gap: {bt['gap'].mean():.2f}  (should be ~0.21)")

weekly = bt.groupby(["season", "week"])["gap"].mean()
print(weekly.describe().round(2))
print(f"Weeks with gap >= 1.04: {(weekly >= 1.04).mean():.0%}")

Overall gap: 0.21  (should be ~0.21)
count    236.00
mean       0.21
std        0.90
min       -3.04
25%       -0.30
50%        0.21
75%        0.70
max        4.18
Name: gap, dtype: float64
Weeks with gap >= 1.04: 13%


## How similar are the models?

Predictions and, more importantly, errors are highly correlated across models, which is why
combining them adds little.

In [2]:
import pandas as pd
from nflmodel.features.build import build_features
from nflmodel.models.elo import EloModel
from nflmodel.models.ridge import RidgeModel
from nflmodel.models.lgbm import LGBMModel
from nflmodel.models.massey import MasseyModel
from nflmodel.config import BACKTEST_SEASONS

models = {
    "elo": EloModel(),
    "massey": MasseyModel(),
    "ridge": RidgeModel(),
    "lgbm": LGBMModel(),
}

games = build_features()
rows = []
for season in BACKTEST_SEASONS:
    train = games[games["season"] < season]
    test = games[games["season"] == season].copy()
    for name, model in models.items():
        test[name] = model.fit(train).predict(test)
    rows.append(test)
preds = pd.concat(rows)

# how similar are the predictions? (1.0 = identical)
print(preds[["ridge", "lgbm", "elo", "massey"]].corr().round(3))

# how similar are the MISTAKES? This is what matters for combining models
errors = preds[["ridge", "lgbm", "elo", "massey"]].sub(preds["result"], axis=0)
print(errors.corr().round(3))

        ridge   lgbm    elo  massey
ridge   1.000  0.966  0.944   0.951
lgbm    0.966  1.000  0.947   0.950
elo     0.944  0.947  1.000   0.939
massey  0.951  0.950  0.939   1.000
        ridge   lgbm    elo  massey
ridge   1.000  0.993  0.989   0.990
lgbm    0.993  1.000  0.991   0.990
elo     0.989  0.991  1.000   0.988
massey  0.990  0.990  0.988   1.000


## Ridge weights

What the live ridge model leans on (features are standardized, so weights are comparable).

In [4]:
import pandas as pd
from nflmodel.features.build import build_features
from nflmodel.models.ridge import RidgeModel

df = build_features()
model = RidgeModel().fit(df)              # all completed games, same as the live model

ridge = model.pipe[-1]                    # the Ridge step inside the pipeline
weights = pd.Series(ridge.coef_, index=model.features)
print(weights.sort_values(key=abs, ascending=False).round(2))
print("intercept:", round(ridge.intercept_, 2))

massey_margin         2.50
elo_margin            1.77
home_off_epa_prior    0.93
away_qb_starts       -0.87
away_off_epa_prior   -0.83
home_qb_starts        0.82
home_def_epa_prior   -0.69
rest_diff             0.38
away_def_epa_prior   -0.23
home_field            0.21
dtype: float64
intercept: 2.13


## Does blending ridge with LightGBM help?

No: the best blend improves MAE by 0.002 points.

In [6]:
import pandas as pd
from nflmodel.features.build import build_features
from nflmodel.models.ridge import RidgeModel
from nflmodel.models.lgbm import LGBMModel
from nflmodel.config import BACKTEST_SEASONS

games = build_features()
rows = []
for season in BACKTEST_SEASONS:
    train = games[games["season"] < season]
    test = games[games["season"] == season].copy()
    test["ridge"] = RidgeModel().fit(train).predict(test)
    test["lgbm"] = LGBMModel().fit(train).predict(test)
    rows.append(test)
preds = pd.concat(rows)

for w in [1.0, 0.8, 0.7, 0.5]:
    blend = w * preds["ridge"] + (1 - w) * preds["lgbm"]
    print(f"ridge weight {w}: MAE {(blend - preds['result']).abs().mean():.3f}")

ridge weight 1.0: MAE 10.067
ridge weight 0.8: MAE 10.065
ridge weight 0.7: MAE 10.065
ridge weight 0.5: MAE 10.071
